# 07 Attention Mechanisms: Bahdanau, Luong & Bridge to Transformers

**Real-World Scenario**: Enterprise Multilingual Technical Support Alignment & Query Attention System.

This notebook demonstrates building a PyTorch Luong Dot-Product Attention module, computing alignment weights $\alpha_{ij}$ step-by-step, generating alignment heatmap visualizations, saving artifacts into a hidden `.model_cache/` directory, and executing a **Complete GenAI Multilingual Support LLM Call**.

## Step 1: Environment Setup & Local Cache Configuration

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from root .env file
load_dotenv()

# Create hidden model cache directory for local pipeline artifact persistence
os.makedirs(".model_cache", exist_ok=True)

print("=== Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden Cache Directory Path:", os.path.abspath(".model_cache"))

## Step 2: Step-by-Step Luong Dot-Product Attention Hand-Calculation

In [ ]:
import numpy as np

# Encoder Hidden States H (3 tokens, d=2)
h1 = np.array([1.0, 0.0])
h2 = np.array([0.0, 2.0])
h3 = np.array([1.0, 1.0])
H = np.vstack([h1, h2, h3]) # [3, 2]

# Decoder Hidden State s_i (d=2)
s_i = np.array([2.0, 1.0]) # [2,]

# 1. Compute Dot Attention Scores e_j = s_i^T * h_j
scores = np.dot(H, s_i) # [3,]

# 2. Compute Softmax Alignment Probabilities
exp_scores = np.exp(scores - np.max(scores)) # Stability adjustment
alpha = exp_scores / np.sum(exp_scores)

# 3. Compute Context Vector c_i = sum(alpha_j * h_j)
context_vec = np.sum(H * alpha[:, np.newaxis], axis=0)

print("=== Luong Dot Attention Hand-Calculation Results ===")
print("Raw Scores e_j:            ", scores)
print("Alignment Weights alpha:   ", alpha.round(4))
print("Dynamic Context Vector c_i:", context_vec.round(4))

## Step 3: PyTorch Luong Attention Module Implementation & Weights Persistence

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PyTorchLuongAttention(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, decoder_state, encoder_states):
        # decoder_state: [batch, 1, hidden]
        # encoder_states: [batch, seq_len, hidden]
        scores = torch.bmm(decoder_state, encoder_states.transpose(1, 2))
        weights = F.softmax(scores, dim=-1)
        context = torch.bmm(weights, encoder_states)
        return context, weights

attn_module = PyTorchLuongAttention()
model_path = os.path.join(".model_cache", "luong_attention.pt")
torch.save(attn_module.state_dict(), model_path)
print(f"=== PyTorch Attention Module Saved to Hidden Folder: {model_path} ===")

## Step 4: Batch Alignment Weights Matrix Execution & Heatmap Analysis

In [ ]:
batch_size, seq_len, hidden_dim = 1, 4, 8
encoder_h = torch.randn(batch_size, seq_len, hidden_dim)
decoder_s = torch.randn(batch_size, 1, hidden_dim)

context_tensor, weights_tensor = attn_module(decoder_s, encoder_h)
weights_np = weights_tensor.squeeze(0).detach().numpy()

source_tokens = ["database", "connection", "failover", "timeout"]
print("=== Attention Alignment Weights Distribution ===")
for tok, w in zip(source_tokens, weights_np[0]):
    print(f"Token: {tok:<15} | Attention Weight: {w*100:.1f}%")

## Step 5: Executing Complete GenAI Multilingual Support LLM Call

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

top_attended_token = source_tokens[np.argmax(weights_np[0])]

if os.getenv("OPENAI_API_KEY"):
    prompt = ChatPromptTemplate.from_template('''You are an Enterprise Multilingual Support AI.
An attention alignment module focused highest weight on token: '{token}' in query: '{query}'.
Provide a concise solution summary for the support customer.

Support Solution:''')
    
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    full_query = "database connection failover timeout"
    
    response = llm.invoke(prompt.format(token=top_attended_token, query=full_query))
    print("=== Complete GenAI Multilingual Support Response ===")
    print(response.content)
else:
    print("[NOTICE] OPENAI_API_KEY not found in .env. Skipping live LLM call.")